# 06 - NetworkX Integration and Visualization

Orthograph integrates with NetworkX for graph operations and provides Mermaid diagram generation for schema visualization. This notebook covers:

- Converting a model schema to a NetworkX graph for analysis
- Validating data stored in a NetworkX `MultiDiGraph`
- Detecting errors in NetworkX graph data
- Generating Mermaid diagrams from model definitions

In [1]:
from typing import Optional

import networkx as nx

from orthograph import (
    GraphDataModel,
    NodeModel,
    RelationshipModel,
    Cardinality,
)
from orthograph.extensions.networkx import schema_to_networkx, validate_networkx_graph
from orthograph.depiction import to_mermaid

## Define the model

We use a chemistry domain: molecules participate as reactants or products in chemical equations, and each equation can have a reaction template. This is a common pattern in cheminformatics knowledge graphs.

In [2]:
class Molecule(NodeModel):
    __label__ = "Molecule"
    __uid_field__ = "uid"
    uid: str
    smiles: str


class ChemicalEquation(NodeModel):
    __label__ = "ChemicalEquation"
    __uid_field__ = "uid"
    uid: str
    smiles: str


class Template(NodeModel):
    __label__ = "Template"
    __uid_field__ = "uid"
    uid: str
    smarts: str


class Reactant(RelationshipModel):
    __label__ = "REACTANT"
    __source_type__ = Molecule
    __target_type__ = ChemicalEquation
    __source_cardinality__ = Cardinality.ZERO_OR_MORE
    __target_cardinality__ = Cardinality.ONE_OR_MORE


class Product(RelationshipModel):
    __label__ = "PRODUCT"
    __source_type__ = ChemicalEquation
    __target_type__ = Molecule
    __source_cardinality__ = Cardinality.ONE_OR_MORE
    __target_cardinality__ = Cardinality.ZERO_OR_MORE


class HasTemplate(RelationshipModel):
    __label__ = "HAS_TEMPLATE"
    __source_type__ = ChemicalEquation
    __target_type__ = Template
    __source_cardinality__ = Cardinality.ZERO_OR_ONE


model = GraphDataModel(
    name="Chemistry",
    node_types=[Molecule, ChemicalEquation, Template],
    relationship_types=[Reactant, Product, HasTemplate],
)

print("Model:", model.name)
print("Nodes:", model.node_labels)
print("Rels: ", model.relationship_labels)

Model: Chemistry
Nodes: {'Template', 'Molecule', 'ChemicalEquation'}
Rels:  {'PRODUCT', 'HAS_TEMPLATE', 'REACTANT'}


## Schema as a NetworkX graph

The `schema_to_networkx` function converts the model definition itself -- not instance data -- into a NetworkX `MultiDiGraph`. Nodes in this graph represent node types; edges represent relationship types. This is useful for programmatic analysis of the schema structure.

In [3]:
schema_graph = schema_to_networkx(model)

print("Schema graph nodes:")
for node, attrs in schema_graph.nodes(data=True):
    print(f"  {node}: uid_field={attrs['uid_field']}, properties={attrs['properties']}")

print()
print("Schema graph edges:")
for src, tgt, attrs in schema_graph.edges(data=True):
    print(f"  {src} --[{attrs['label']}]--> {tgt}")
    print(f"    source_cardinality={attrs['source_cardinality']}, target_cardinality={attrs['target_cardinality']}")

Schema graph nodes:
  Molecule: uid_field=uid, properties={'uid': 'str', 'smiles': 'str'}
  ChemicalEquation: uid_field=uid, properties={'uid': 'str', 'smiles': 'str'}
  Template: uid_field=uid, properties={'uid': 'str', 'smarts': 'str'}

Schema graph edges:
  Molecule --[REACTANT]--> ChemicalEquation
    source_cardinality=min=0 max=None, target_cardinality=min=1 max=None
  ChemicalEquation --[PRODUCT]--> Molecule
    source_cardinality=min=1 max=None, target_cardinality=min=0 max=None
  ChemicalEquation --[HAS_TEMPLATE]--> Template
    source_cardinality=min=0 max=1, target_cardinality=min=0 max=None


## Validating a NetworkX graph

If your data already lives in a NetworkX `MultiDiGraph`, you can validate it directly using `validate_networkx_graph`. Each node must have a `__label__` attribute and the relevant properties. Edges must have `__label__` and any required properties.

In [4]:
# Build a small chemistry graph in NetworkX
G = nx.MultiDiGraph()

# Nodes: use the node ID as the graph key, store properties as attributes
G.add_node("mol_1", __label__="Molecule", uid="mol_1", smiles="CCO")
G.add_node("mol_2", __label__="Molecule", uid="mol_2", smiles="CC=O")
G.add_node("ce_1",  __label__="ChemicalEquation", uid="ce_1", smiles="CCO>>CC=O")
G.add_node("tpl_1", __label__="Template", uid="tpl_1", smarts="[C:1][OH]>>[C:1]=O")

# Edges: REACTANT, PRODUCT, HAS_TEMPLATE
G.add_edge("mol_1", "ce_1", __label__="REACTANT")
G.add_edge("ce_1", "mol_2", __label__="PRODUCT")
G.add_edge("ce_1", "tpl_1", __label__="HAS_TEMPLATE")

# Validate
result = validate_networkx_graph(G, model)
print("Valid:", result.is_valid)
print("Errors:", len(result.errors))
print("Warnings:", len(result.warnings))

Valid: True
Errors: 0
Warnings: 0


## Catching errors in NetworkX graphs

When the graph contains data that does not conform to the model -- such as unknown labels, missing properties, or type mismatches -- the validator reports detailed issues.

In [5]:
# Add an invalid node: unknown label 'Catalyst'
G.add_node("cat_1", __label__="Catalyst", uid="cat_1", name="Pd/C")

# Add a node missing a required property (Molecule without 'smiles')
G.add_node("mol_bad", __label__="Molecule", uid="mol_bad")

result = validate_networkx_graph(G, model)
print("Valid:", result.is_valid)
print(f"Found {len(result.errors)} error(s):\n")
for issue in result.errors:
    print(f"  [{issue.code}] {issue.message}")

# Clean up the invalid nodes for the next cells
G.remove_node("cat_1")
G.remove_node("mol_bad")

Valid: False
Found 2 error(s):

  [UNKNOWN_NODE_LABEL] Unknown node label: Catalyst
  [PROPERTY_VALIDATION_ERROR] Validation error: Field required (field: smiles)


## Mermaid diagram generation

The `to_mermaid` function generates a text-based Mermaid diagram of the model schema. This can be rendered in any Mermaid-compatible viewer (GitHub markdown, mermaid.live, Jupyter with mermaid extensions, etc.).

In [6]:
mermaid_text = to_mermaid(model)
print(mermaid_text)

graph TD
    Molecule["Molecule<br>uid: str, smiles: str"]
    ChemicalEquation["ChemicalEquation<br>uid: str, smiles: str"]
    Template["Template<br>uid: str, smarts: str"]
    Molecule -->|REACTANT| ChemicalEquation
    ChemicalEquation -->|PRODUCT| Molecule
    ChemicalEquation -->|HAS_TEMPLATE| Template


The Mermaid diagram below will render in viewers that support `mermaid` fenced code blocks (GitHub, mermaid.live, Jupyter with appropriate extensions).

```mermaid
graph TD
    Molecule["Molecule<br>uid: str, smiles: str"]
    ChemicalEquation["ChemicalEquation<br>uid: str, smiles: str"]
    Template["Template<br>uid: str, smarts: str"]
    Molecule -->|REACTANT| ChemicalEquation
    ChemicalEquation -->|PRODUCT| Molecule
    ChemicalEquation -->|HAS_TEMPLATE| Template
```